# 02 — Exploratory Data Analysis

The PRD asks for at least 20 visualizations. The following cells cover demographics, plan/revenue mix, churn, engagement, billing, support, and feedback.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
D=ROOT/"data/processed"
customers=pd.read_csv(D/"customers.csv",parse_dates=["registration_date"])
subs=pd.read_csv(D/"subscriptions.csv",parse_dates=["subscription_start_date","subscription_end_date","cancellation_date"])
view=pd.read_csv(D/"viewing_activity.csv",parse_dates=["viewing_date"])
pay=pd.read_csv(D/"payments.csv",parse_dates=["payment_date"])
tickets=pd.read_csv(D/"support_tickets.csv",parse_dates=["ticket_date"])
feedback=pd.read_csv(D/"customer_feedback.csv",parse_dates=["feedback_date"])
content=pd.read_csv(D/"content.csv")
features=pd.read_csv(D/"customer_features.csv")


### 1. Customers by country

In [ ]:
fig=px.bar(customers.country.value_counts().head(20).reset_index(),x="country",y="count",title="Customers by Country")
fig.show()

### 2. Age distribution

In [ ]:
fig=px.histogram(customers,x="age",nbins=35,title="Customer Age Distribution")
fig.show()

### 3. Acquisition channel

In [ ]:
fig=px.bar(customers.acquisition_channel.value_counts().reset_index(),x="acquisition_channel",y="count",title="Acquisition Channel")
fig.show()

### 4. Customer segment

In [ ]:
fig=px.pie(customers,names="customer_segment",title="Customer Segment Mix")
fig.show()

### 5. Current plan mix

In [ ]:
fig=px.pie(features,names="current_plan",title="Current Plan Mix")
fig.show()

### 6. Revenue by plan

In [ ]:
fig=px.bar(features.groupby("current_plan",as_index=False).current_monthly_price.sum(),x="current_plan",y="current_monthly_price",title="Monthly Recurring Revenue Proxy by Plan")
fig.show()

### 7. Churn by plan

In [ ]:
fig=px.bar(features.groupby("current_plan",as_index=False).churned.mean().assign(churn_pct=lambda x:100*x.churned),x="current_plan",y="churn_pct",title="Churn Rate by Plan")
fig.show()

### 8. Churn by country

In [ ]:
fig=px.bar(features.groupby("country",as_index=False).churned.mean().assign(churn_pct=lambda x:100*x.churned).sort_values("churn_pct",ascending=False).head(20),x="country",y="churn_pct",title="Churn Rate by Country")
fig.show()

### 9. Churn by tenure

In [ ]:
fig=px.histogram(features,x="customer_tenure_days",color="churned",barmode="overlay",nbins=30,title="Tenure vs Churn")
fig.show()

### 10. Churn by engagement

In [ ]:
fig=px.box(features,x="churned",y="engagement_score",title="Engagement Score vs Churn")
fig.show()

### 11. Watch minutes

In [ ]:
fig=px.histogram(view,x="watch_duration_minutes",nbins=60,title="Watch Duration",log_y=True)
fig.show()

### 12. Completion distribution

In [ ]:
fig=px.histogram(view,x="completion_percentage",nbins=40,title="Completion Percentage")
fig.show()

### 13. Device usage

In [ ]:
fig=px.bar(view.device_type.value_counts().reset_index(),x="device_type",y="count",title="Viewing Sessions by Device")
fig.show()

### 14. Monthly viewing minutes

In [ ]:
fig=px.line(view.assign(month=view.viewing_date.dt.to_period("M").astype(str)).groupby("month",as_index=False).watch_duration_minutes.sum(),x="month",y="watch_duration_minutes",title="Monthly Watch Minutes")
fig.show()

### 15. Top genres by minutes

In [ ]:
fig=px.bar(view.merge(content[["content_id","genre"]],on="content_id").groupby("genre",as_index=False).watch_duration_minutes.sum().sort_values("watch_duration_minutes",ascending=False),x="genre",y="watch_duration_minutes",title="Watch Minutes by Genre")
fig.show()

### 16. Failed payments by month

In [ ]:
fig=px.line(pay.assign(month=pay.payment_date.dt.to_period("M").astype(str)).query("payment_status == 'Failed'").groupby("month",as_index=False).size(),x="month",y="size",title="Failed Payments by Month")
fig.show()

### 17. Payment amount distribution

In [ ]:
fig=px.histogram(pay,x="amount",color="payment_status",nbins=50,title="Payment Amount Distribution",log_y=True)
fig.show()

### 18. Tickets by category

In [ ]:
fig=px.bar(tickets.issue_category.value_counts().reset_index(),x="issue_category",y="count",title="Support Tickets by Category")
fig.show()

### 19. Resolution time by category

In [ ]:
fig=px.box(tickets,x="issue_category",y="resolution_time_hours",title="Resolution Time by Category")
fig.show()

### 20. CSAT distribution

In [ ]:
fig=px.histogram(tickets,x="customer_satisfaction_score",title="Support CSAT")
fig.show()

### 21. Complaint subcategories

In [ ]:
fig=px.bar(tickets.issue_subcategory.value_counts().head(15).reset_index(),x="issue_subcategory",y="count",title="Top Complaint Subcategories")
fig.show()

### 22. Feedback sentiment/rating

In [ ]:
fig=px.box(feedback,x="sentiment_label",y="rating",title="Feedback Rating by Sentiment Label")
fig.show()

### 23. Failed payment rate vs churn

In [ ]:
fig=px.box(features,x="churned",y="failed_payment_rate",title="Failed Payment Rate vs Churn")
fig.show()

### 24. Support volume vs churn

In [ ]:
fig=px.box(features,x="churned",y="support_ticket_count",title="Support Ticket Volume vs Churn")
fig.show()

### 25. Engagement vs CLV

In [ ]:
fig=px.scatter(features.sample(min(3000,len(features)),random_state=42),x="engagement_score",y="customer_lifetime_value",color="churned",title="Engagement vs Customer Lifetime Value",hover_name="customer_id")
fig.show()